# Coarse Consistency Global Ambiguity

This notebook builds two presentation-oriented artifact families from an existing coarse-consistency output directory:

- a `fig1c`-style global qualitative variant with rows `Generated`, `Recoarsened`, `GT`, and `Rel. Error`
- standalone tree-layout tiles for the message "one coarse observation, many compatible fine fields"

The notebook reuses the saved coarse-consistency metrics/cache contract and the existing transfer operator implementation.

In [1]:
from __future__ import annotations

import csv
import json
import os
import sys
from pathlib import Path
from typing import Any

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

if "__file__" in globals():
    REPO_ROOT = Path(__file__).resolve().parent.parent
else:
    cwd = Path.cwd().resolve()
    if (cwd / "mmsfm").exists():
        REPO_ROOT = cwd
    elif (cwd.parent / "mmsfm").exists():
        REPO_ROOT = cwd.parent
    else:
        REPO_ROOT = cwd

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.fae.tran_evaluation.coarse_consistency_artifacts import load_saved_coarse_report_payload
from scripts.fae.tran_evaluation.core import FilterLadder
from scripts.fae.tran_evaluation import report as tran_report
from scripts.images.field_visualization import format_for_paper

format_for_paper()


def _parse_float_list(text: str) -> list[float]:
    values = []
    for item in str(text).split(","):
        stripped = item.strip()
        if not stripped:
            continue
        values.append(float(stripped))
    return values


def _format_h_slug(value: float) -> str:
    numeric = float(value)
    if numeric.is_integer():
        return str(int(numeric))
    return f"{numeric:g}".replace("-", "m").replace(".", "p")


print(f"REPO_ROOT: {REPO_ROOT}")

REPO_ROOT: /data1/jy384/research/MMSFM


/home/jy384/miniconda3/envs/3MASB/lib/python3.13/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/home/jy384/miniconda3/envs/3MASB/lib/python3.13/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened becau

## Configuration

All settings can be overridden via environment variables before execution.

- `COARSE_OUTPUT_DIR`
- `EXPORT_DIR`
- `TREE_SOURCE_HS`
- `N_BRANCHES`
- `REL_ERROR_EPS`
- `REL_ERROR_CLIP_PERCENTILE`
- `TRANSPARENT_EXPORT`
- `FIG_DPI`
- `TILE_INCHES`

In [2]:
DEFAULT_COARSE_OUTPUT_DIR = REPO_ROOT / "results" / "csp" / "transformer_patch8_adamw_ntk_prior_balanced_l128x128_prior128h3h4_logsnr5_token_dit_set_conditioned_memory" / "main" / "eval" / "n512" / "generated_consistency_m100_r100_k6_gpu_resume_20260419_214336"

COARSE_OUTPUT_DIR = Path(
    os.environ.get("COARSE_OUTPUT_DIR", str(DEFAULT_COARSE_OUTPUT_DIR))
).expanduser().resolve()
EXPORT_DIR = Path(
    os.environ.get(
        "EXPORT_DIR",
        str(REPO_ROOT / "notebooks" / "figures" / "coarse_consistency_global_ambiguity" / COARSE_OUTPUT_DIR.name),
    )
).expanduser().resolve()
TREE_SOURCE_HS = _parse_float_list(os.environ.get("TREE_SOURCE_HS", "4.0,2.0,1.0"))
N_BRANCHES = int(os.environ.get("N_BRANCHES", "3"))
REL_ERROR_EPS = float(os.environ.get("REL_ERROR_EPS", "1e-3"))
REL_ERROR_CLIP_PERCENTILE = float(os.environ.get("REL_ERROR_CLIP_PERCENTILE", "99.5"))
TRANSPARENT_EXPORT = os.environ.get("TRANSPARENT_EXPORT", "1") != "0"
FIG_DPI = int(os.environ.get("FIG_DPI", "300"))
TILE_INCHES = float(os.environ.get("TILE_INCHES", "1.6"))

EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"COARSE_OUTPUT_DIR: {COARSE_OUTPUT_DIR}")
print(f"EXPORT_DIR       : {EXPORT_DIR}")
print(f"TREE_SOURCE_HS   : {TREE_SOURCE_HS}")
print(f"N_BRANCHES       : {N_BRANCHES}")
print(f"REL_ERROR_EPS    : {REL_ERROR_EPS}")
print(f"REL_ERR_CLIP_PCT : {REL_ERROR_CLIP_PERCENTILE}")
print(f"TRANSPARENT      : {TRANSPARENT_EXPORT}")

COARSE_OUTPUT_DIR: /data1/jy384/research/MMSFM/results/csp/transformer_patch8_adamw_ntk_prior_balanced_l128x128_prior128h3h4_logsnr5_token_dit_set_conditioned_memory/main/eval/n512/generated_consistency_m100_r100_k6_gpu_resume_20260419_214336
EXPORT_DIR       : /data1/jy384/research/MMSFM/notebooks/figures/coarse_consistency_global_ambiguity/generated_consistency_m100_r100_k6_gpu_resume_20260419_214336
TREE_SOURCE_HS   : [4.0, 2.0, 1.0]
N_BRANCHES       : 3
REL_ERROR_EPS    : 0.001
REL_ERR_CLIP_PCT : 99.5
TRANSPARENT      : True


In [3]:
def _field_to_image(field: np.ndarray, resolution: int) -> np.ndarray:
    arr = np.asarray(field, dtype=np.float64)
    if arr.ndim == 1:
        return arr.reshape(resolution, resolution)
    if arr.ndim == 2 and arr.shape == (resolution, resolution):
        return arr
    raise ValueError(f"Expected flattened or square image field, got {arr.shape}.")


def _save_figure(fig: plt.Figure, base_path: Path, *, transparent: bool = TRANSPARENT_EXPORT) -> None:
    base_path.parent.mkdir(parents=True, exist_ok=True)
    fig.patch.set_alpha(0.0 if transparent else 1.0)
    fig.savefig(base_path.with_suffix(".png"), dpi=FIG_DPI, bbox_inches="tight", transparent=transparent)
    fig.savefig(base_path.with_suffix(".pdf"), bbox_inches="tight", transparent=transparent)
    plt.close(fig)


def _save_tile(image: np.ndarray, *, cmap: str, vmin: float, vmax: float, base_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(TILE_INCHES, TILE_INCHES))
    ax.imshow(np.asarray(image), origin="lower", cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    _save_figure(fig, base_path)


def _save_vertical_colorbar(*, vmin: float, vmax: float, cmap: str, base_path: Path, label: str | None = None) -> None:
    fig, ax = plt.subplots(figsize=(0.85, 2.4))
    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar = fig.colorbar(sm, cax=ax)
    cbar.ax.tick_params(labelsize=tran_report.FONT_TICK)
    if label:
        cbar.set_label(label, fontsize=tran_report.FONT_LABEL)
    _save_figure(fig, base_path)


def _robust_symmetric_limits(images: list[np.ndarray], percentile: float) -> tuple[float, float]:
    if not images:
        return (-1.0, 1.0)
    flat = np.concatenate([np.abs(np.asarray(img, dtype=np.float64)).ravel() for img in images], axis=0)
    vmax = float(np.nanpercentile(flat, percentile)) if flat.size else 1.0
    if not np.isfinite(vmax) or vmax <= 0.0:
        vmax = float(np.nanmax(flat)) if flat.size else 1.0
    if not np.isfinite(vmax) or vmax <= 0.0:
        vmax = 1.0
    return (-vmax, vmax)


def _signed_relative_error_map(recoarsened: np.ndarray, gt: np.ndarray, rel_eps: float) -> np.ndarray:
    recoarsened_arr = np.asarray(recoarsened, dtype=np.float64)
    gt_arr = np.asarray(gt, dtype=np.float64)
    gt_scale = float(np.max(np.abs(gt_arr)))
    if gt_scale <= 0.0:
        denom = np.ones_like(gt_arr, dtype=np.float64)
    else:
        denom = np.maximum(np.abs(gt_arr), float(rel_eps) * gt_scale)
    return (recoarsened_arr - gt_arr) / denom


def _resolve_rollout_index(source_h: float, modeled_h_schedule: list[float]) -> int:
    for idx, h_value in enumerate(modeled_h_schedule):
        if np.isclose(float(h_value), float(source_h), atol=1e-9, rtol=1e-9):
            return int(idx)
    raise KeyError(f"Could not find source H={source_h:g} in modeled schedule {modeled_h_schedule}.")


def _load_global_cache_small(output_dir: Path) -> dict[str, np.ndarray]:
    cache_path = Path(output_dir) / "cache" / "conditioned_global.npz"
    with np.load(cache_path, allow_pickle=True) as data:
        return {
            "test_sample_indices": np.asarray(data["test_sample_indices"], dtype=np.int64),
            "time_indices": np.asarray(data["time_indices"], dtype=np.int64),
        }


def _load_selected_global_rollout_condition(output_dir: Path, condition_index: int) -> dict[str, Any]:
    cache_root = Path(output_dir) / "cache"
    decoded_store_dir = cache_root / "conditioned_global.cache"
    if decoded_store_dir.exists():
        status_path = decoded_store_dir / "status.json"
        if status_path.exists():
            status = json.loads(status_path.read_text())
            chunk_size = int(status.get("chunk_size", 1))
            chunk_start = (int(condition_index) // int(chunk_size)) * int(chunk_size)
            chunk_name = f"condition_chunk_{int(chunk_start):06d}"
            chunk_path = decoded_store_dir / "chunks" / f"{chunk_name}.npz"
            local_index = int(condition_index) - int(chunk_start)
            with np.load(chunk_path, allow_pickle=True) as chunk:
                return {
                    "decoded_rollout_fields": np.asarray(chunk["decoded_rollout_fields"][local_index], dtype=np.float32),
                    "decoded_finest_fields": np.asarray(chunk["decoded_finest_fields"][local_index], dtype=np.float32),
                    "coarse_target": np.asarray(chunk["coarse_targets"][local_index], dtype=np.float32),
                    "source": str(chunk_path),
                    "chunk_start": int(chunk_start),
                    "local_index": int(local_index),
                }
    cache_path = cache_root / "conditioned_global.npz"
    with np.load(cache_path, allow_pickle=True) as data:
        return {
            "decoded_rollout_fields": np.asarray(data["decoded_rollout_fields"][condition_index], dtype=np.float32),
            "decoded_finest_fields": np.asarray(data["decoded_finest_fields"][condition_index], dtype=np.float32),
            "coarse_target": np.asarray(data["coarse_targets"][condition_index], dtype=np.float32),
            "source": str(cache_path),
            "chunk_start": None,
            "local_index": int(condition_index),
        }


def _write_manifest_rows(rows: list[dict[str, Any]], *, json_path: Path, csv_path: Path, summary: dict[str, Any]) -> None:
    json_path.write_text(json.dumps({"summary": summary, "rows": rows}, indent=2, sort_keys=True))
    fieldnames = sorted({key for row in rows for key in row.keys()}) if rows else []
    with csv_path.open("w", newline="") as handle:
        if fieldnames:
            writer = csv.DictWriter(handle, fieldnames=fieldnames)
            writer.writeheader()
            for row in rows:
                writer.writerow(row)


def _build_global_relative_error_panel(*, qualitative_examples: dict[str, Any], resolution: int, export_dir: Path) -> dict[str, Any]:
    generated_fields = np.asarray(qualitative_examples["generated_fields"], dtype=np.float32)
    recoarsened_fields = np.asarray(qualitative_examples["coarsened_fields"], dtype=np.float32)
    condition_fields = np.asarray(qualitative_examples["condition_fields"], dtype=np.float32)
    n_cols = min(int(generated_fields.shape[0]), int(recoarsened_fields.shape[0]), int(condition_fields.shape[0]))
    generated_imgs = [_field_to_image(generated_fields[idx], resolution) for idx in range(n_cols)]
    recoarsened_imgs = [_field_to_image(recoarsened_fields[idx], resolution) for idx in range(n_cols)]
    gt_imgs = [_field_to_image(condition_fields[idx], resolution) for idx in range(n_cols)]
    relative_error_imgs = [
        _signed_relative_error_map(recoarsened_imgs[idx], gt_imgs[idx], REL_ERROR_EPS)
        for idx in range(n_cols)
    ]
    recoarsened_gt_limits = tran_report._image_limits(recoarsened_imgs + gt_imgs)

    row_specs = [
        {"label": "Generated", "images": generated_imgs, "cmap": tran_report.CMAP_FIELD, "limits": tran_report._image_limits(generated_imgs)},
        {"label": "Recoarsened", "images": recoarsened_imgs, "cmap": tran_report.CMAP_FIELD, "limits": recoarsened_gt_limits},
        {"label": "GT", "images": gt_imgs, "cmap": tran_report.CMAP_FIELD, "limits": recoarsened_gt_limits},
        {"label": "Rel. Error", "images": relative_error_imgs, "cmap": tran_report.CMAP_DIFF, "limits": _robust_symmetric_limits(relative_error_imgs, REL_ERROR_CLIP_PERCENTILE)},
    ]

    fig, axes = plt.subplots(
        len(row_specs),
        n_cols,
        figsize=tran_report._publication_conditioned_panel_size(n_rows=len(row_specs), n_cols=n_cols, rowwise_colorbars=True),
        squeeze=False,
    )
    im_handles = [None] * len(row_specs)
    for row_idx, spec in enumerate(row_specs):
        vmin, vmax = spec["limits"]
        for col_idx in range(n_cols):
            im = axes[row_idx, col_idx].imshow(
                spec["images"][col_idx],
                origin="lower",
                cmap=spec["cmap"],
                vmin=vmin,
                vmax=vmax,
            )
            if col_idx == 0:
                im_handles[row_idx] = im
            axes[row_idx, col_idx].axis("off")
    fig.subplots_adjust(left=0.12, right=0.86, top=0.985, bottom=0.03, wspace=0.03, hspace=0.08)
    tran_report._add_left_row_labels(fig, axes, [spec["label"] for spec in row_specs], x=0.07, fontsize=tran_report.FONT_LABEL)
    for row_idx, im in enumerate(im_handles):
        tran_report.add_row_cbar_vertical(fig, list(axes[row_idx, :]), im)

    figure_base = export_dir / "fig1c_coarse_consistency_global_qualitative_relative_error"
    _save_figure(fig, figure_base)
    return {
        "figure_base": str(figure_base),
        "n_cols": int(n_cols),
        "recoarsened_gt_limits": [float(item) for item in recoarsened_gt_limits],
        "relative_error_limits": [float(item) for item in row_specs[-1]["limits"]],
    }


def _export_ambiguity_tree_assets(
    *,
    output_dir: Path,
    export_dir: Path,
    resolution: int,
    config: dict[str, Any],
    qualitative_examples: dict[str, Any],
    ladder: FilterLadder,
    tree_source_hs: list[float],
    n_branches: int,
) -> dict[str, Any]:
    global_cache_small = _load_global_cache_small(output_dir)
    full_h_schedule = [float(item) for item in config["full_H_schedule"]]
    time_indices = np.asarray(global_cache_small["time_indices"], dtype=np.int64)
    modeled_h_schedule = [float(full_h_schedule[int(tidx)]) for tidx in time_indices]

    condition_indices = np.asarray(qualitative_examples["condition_indices"], dtype=np.int64)
    realization_indices = np.asarray(qualitative_examples["realization_indices"], dtype=np.int64)
    if condition_indices.size == 0 or realization_indices.size == 0:
        raise ValueError("The saved global qualitative payload did not include branch selections.")
    selected_condition_index = int(condition_indices[0])
    selected_realization_indices = realization_indices[: int(n_branches)].astype(np.int64)
    selected_test_sample_index = int(global_cache_small["test_sample_indices"][selected_condition_index])

    transfer_meta = qualitative_examples.get("transfer_metadata", {}) or {}
    root_h = float(transfer_meta.get("target_H"))
    ridge_lambda = float(transfer_meta.get("ridge_lambda", 0.0) or 0.0)
    operator_name = str(transfer_meta.get("operator_name", "tran_periodic_spectral_transfer"))

    selected_condition_payload = _load_selected_global_rollout_condition(output_dir, selected_condition_index)
    rollout_fields = np.asarray(selected_condition_payload["decoded_rollout_fields"], dtype=np.float32)
    root_gt_img = _field_to_image(selected_condition_payload["coarse_target"], resolution)

    rows: list[dict[str, Any]] = []
    tile_records: list[dict[str, Any]] = []
    for source_h in tree_source_hs:
        rollout_idx = _resolve_rollout_index(float(source_h), modeled_h_schedule)
        for branch_id, realization_index in enumerate(selected_realization_indices, start=1):
            generated_flat = np.asarray(rollout_fields[int(realization_index), int(rollout_idx)], dtype=np.float32)
            recoarsened_flat = np.asarray(
                ladder.transfer_between_H(
                    generated_flat.reshape(1, -1),
                    source_H=float(source_h),
                    target_H=float(root_h),
                    ridge_lambda=float(ridge_lambda),
                ),
                dtype=np.float32,
            )[0]
            generated_img = _field_to_image(generated_flat, resolution)
            recoarsened_img = _field_to_image(recoarsened_flat, resolution)
            relative_error_img = _signed_relative_error_map(recoarsened_img, root_gt_img, REL_ERROR_EPS)
            tile_records.append(
                {
                    "source_h": float(source_h),
                    "source_rollout_index": int(rollout_idx),
                    "branch_id": int(branch_id),
                    "realization_index": int(realization_index),
                    "generated_img": generated_img,
                    "recoarsened_img": recoarsened_img,
                    "relative_error_img": relative_error_img,
                }
            )

    generated_limits_by_h = {
        float(source_h): tran_report._image_limits([record["generated_img"] for record in tile_records if np.isclose(record["source_h"], float(source_h))])
        for source_h in tree_source_hs
    }
    root_scale_limits = tran_report._image_limits([root_gt_img] + [record["recoarsened_img"] for record in tile_records])
    relative_error_limits = _robust_symmetric_limits([record["relative_error_img"] for record in tile_records], REL_ERROR_CLIP_PERCENTILE)

    root_base = export_dir / f"tree_root_gt_H{_format_h_slug(root_h)}"
    _save_tile(root_gt_img, cmap=tran_report.CMAP_FIELD, vmin=root_scale_limits[0], vmax=root_scale_limits[1], base_path=root_base)

    for source_h, limits in generated_limits_by_h.items():
        generated_legend_base = export_dir / f"legend_generated_H{_format_h_slug(source_h)}_colorbar"
        _save_vertical_colorbar(vmin=limits[0], vmax=limits[1], cmap=tran_report.CMAP_FIELD, base_path=generated_legend_base, label=f"H={source_h:g}")
    root_legend_base = export_dir / f"legend_root_field_H{_format_h_slug(root_h)}_colorbar"
    _save_vertical_colorbar(vmin=root_scale_limits[0], vmax=root_scale_limits[1], cmap=tran_report.CMAP_FIELD, base_path=root_legend_base, label=f"H={root_h:g}")
    error_legend_base = export_dir / "legend_relative_error_colorbar"
    _save_vertical_colorbar(vmin=relative_error_limits[0], vmax=relative_error_limits[1], cmap=tran_report.CMAP_DIFF, base_path=error_legend_base, label="Rel. Error")

    for record in tile_records:
        source_h = float(record["source_h"])
        branch_id = int(record["branch_id"])
        generated_base = export_dir / f"tree_H{_format_h_slug(source_h)}_branch{branch_id:02d}_generated"
        recoarsened_base = export_dir / f"tree_H{_format_h_slug(source_h)}_branch{branch_id:02d}_recoarsened_to_H{_format_h_slug(root_h)}"
        rel_error_base = export_dir / f"tree_H{_format_h_slug(source_h)}_branch{branch_id:02d}_relative_error_to_H{_format_h_slug(root_h)}"

        _save_tile(
            record["generated_img"],
            cmap=tran_report.CMAP_FIELD,
            vmin=generated_limits_by_h[source_h][0],
            vmax=generated_limits_by_h[source_h][1],
            base_path=generated_base,
        )
        _save_tile(
            record["recoarsened_img"],
            cmap=tran_report.CMAP_FIELD,
            vmin=root_scale_limits[0],
            vmax=root_scale_limits[1],
            base_path=recoarsened_base,
        )
        _save_tile(
            record["relative_error_img"],
            cmap=tran_report.CMAP_DIFF,
            vmin=relative_error_limits[0],
            vmax=relative_error_limits[1],
            base_path=rel_error_base,
        )

        rows.append(
            {
                "source_h": float(source_h),
                "target_h": float(root_h),
                "source_rollout_index": int(record["source_rollout_index"]),
                "branch_id": int(branch_id),
                "condition_index": int(selected_condition_index),
                "test_sample_index": int(selected_test_sample_index),
                "realization_index": int(record["realization_index"]),
                "operator_name": str(operator_name),
                "ridge_lambda": float(ridge_lambda),
                "cache_source": str(selected_condition_payload["source"]),
                "generated_base": str(generated_base.relative_to(export_dir)),
                "recoarsened_base": str(recoarsened_base.relative_to(export_dir)),
                "relative_error_base": str(rel_error_base.relative_to(export_dir)),
            }
        )

    summary = {
        "selected_condition_index": int(selected_condition_index),
        "selected_test_sample_index": int(selected_test_sample_index),
        "selected_realization_indices": selected_realization_indices.astype(int).tolist(),
        "modeled_h_schedule": [float(item) for item in modeled_h_schedule],
        "tree_source_hs": [float(item) for item in tree_source_hs],
        "target_h": float(root_h),
        "operator_name": str(operator_name),
        "ridge_lambda": float(ridge_lambda),
        "root_gt_base": str(root_base.relative_to(export_dir)),
        "generated_limits_by_h": {str(key): [float(item) for item in value] for key, value in generated_limits_by_h.items()},
        "root_scale_limits": [float(item) for item in root_scale_limits],
        "relative_error_limits": [float(item) for item in relative_error_limits],
        "root_colorbar_base": str(root_legend_base.relative_to(export_dir)),
        "relative_error_colorbar_base": str(error_legend_base.relative_to(export_dir)),
    }
    _write_manifest_rows(
        rows,
        json_path=export_dir / "ambiguity_tree_manifest.json",
        csv_path=export_dir / "ambiguity_tree_manifest.csv",
        summary=summary,
    )
    return summary


## Load Saved Coarse-Consistency Payloads

The notebook uses the maintained saved-report path, but rebuilds only the global qualitative payload so it does not touch the large conditioned-interval export.

In [4]:
report_payload = load_saved_coarse_report_payload(
    output_dir=COARSE_OUTPUT_DIR,
    l_domain=6.0,
    relative_eps=1e-8,
    include_interval=False,
    include_global=True,
)

metrics_payload = report_payload["metrics_payload"]
config = metrics_payload["config"]
coarse_results = report_payload["coarse_results"]
global_qualitative = report_payload["coarse_qualitative_results"]["conditioned_global_return"]
resolution = int(report_payload["resolution"])

full_h_schedule = [float(item) for item in config["full_H_schedule"]]
ladder = FilterLadder(
    H_schedule=full_h_schedule,
    L_domain=float(config.get("l_domain", 6.0)),
    resolution=int(resolution),
)

print(f"resolution: {resolution}")
print(f"full_h_schedule: {full_h_schedule}")
print(f"transfer operator: {config.get('coarse_transfer_operator')}")
print(f"selected condition indices: {np.asarray(global_qualitative['condition_indices']).tolist()}")
print(f"selected realization indices: {np.asarray(global_qualitative['realization_indices']).tolist()}")

resolution: 128
full_h_schedule: [0.0, 1.0, 1.25, 1.5, 2.0, 2.5, 3.0, 4.0, 6.0]
transfer operator: {'name': 'tran_periodic_tikhonov_transfer', 'formula': 'K_H_target conj(K_H_source) / (|K_H_source|^2 + lambda)', 'ridge_lambda': 1e-08, 'description': 'Regularized periodic kernel-ratio transfer between stored Tran scales.'}
selected condition indices: [24, 24, 24]
selected realization indices: [87, 26, 88]


## Export `fig1c` Relative-Error Variant

In [5]:
global_variant_summary = _build_global_relative_error_panel(
    qualitative_examples=global_qualitative,
    resolution=resolution,
    export_dir=EXPORT_DIR,
)
print(json.dumps(global_variant_summary, indent=2, sort_keys=True))

{
  "figure_base": "/data1/jy384/research/MMSFM/notebooks/figures/coarse_consistency_global_ambiguity/generated_consistency_m100_r100_k6_gpu_resume_20260419_214336/fig1c_coarse_consistency_global_qualitative_relative_error",
  "n_cols": 3,
  "recoarsened_gt_limits": [
    138.63949584960938,
    419.2324523925781
  ],
  "relative_error_limits": [
    -0.11359014130375063,
    0.11359014130375063
  ]
}


## Export Tree-Layout Ambiguity Assets

In [6]:
tree_summary = _export_ambiguity_tree_assets(
    output_dir=COARSE_OUTPUT_DIR,
    export_dir=EXPORT_DIR,
    resolution=resolution,
    config=config,
    qualitative_examples=global_qualitative,
    ladder=ladder,
    tree_source_hs=TREE_SOURCE_HS,
    n_branches=N_BRANCHES,
)
print(json.dumps(tree_summary, indent=2, sort_keys=True))

{
  "generated_limits_by_h": {
    "1.0": [
      0.9999990463256836,
      999.9999389648438
    ],
    "2.0": [
      15.714594841003418,
      815.345947265625
    ],
    "4.0": [
      93.5067367553711,
      472.5150146484375
    ]
  },
  "modeled_h_schedule": [
    1.0,
    1.5,
    2.0,
    3.0,
    4.0,
    6.0
  ],
  "operator_name": "tran_periodic_tikhonov_transfer",
  "relative_error_colorbar_base": "legend_relative_error_colorbar",
  "relative_error_limits": [
    -0.10100131910903645,
    0.10100131910903645
  ],
  "ridge_lambda": 1e-08,
  "root_colorbar_base": "legend_root_field_H6_colorbar",
  "root_gt_base": "tree_root_gt_H6",
  "root_scale_limits": [
    138.63949584960938,
    419.2324523925781
  ],
  "selected_condition_index": 24,
  "selected_realization_indices": [
    87,
    26,
    88
  ],
  "selected_test_sample_index": 314,
  "target_h": 6.0,
  "tree_source_hs": [
    4.0,
    2.0,
    1.0
  ]
}


## Export Summary

In [7]:
exported_files = sorted(path.relative_to(EXPORT_DIR).as_posix() for path in EXPORT_DIR.glob("*") if path.is_file())
print(f"Exported {len(exported_files)} files under {EXPORT_DIR}")
for rel_path in exported_files:
    print(rel_path)

Exported 71 files under /data1/jy384/research/MMSFM/notebooks/figures/coarse_consistency_global_ambiguity/generated_consistency_m100_r100_k6_gpu_resume_20260419_214336
ambiguity_tree_manifest.csv
ambiguity_tree_manifest.json
coarse_consistency_ambiguity_tree_pdfs.zip
fig1c_coarse_consistency_global_qualitative_relative_error.pdf
fig1c_coarse_consistency_global_qualitative_relative_error.png
legend_generated_H1_colorbar.pdf
legend_generated_H1_colorbar.png
legend_generated_H2_colorbar.pdf
legend_generated_H2_colorbar.png
legend_generated_H4_colorbar.pdf
legend_generated_H4_colorbar.png
legend_relative_error_colorbar.pdf
legend_relative_error_colorbar.png
legend_root_field_H6_colorbar.pdf
legend_root_field_H6_colorbar.png
tree_H1_branch01_generated.pdf
tree_H1_branch01_generated.png
tree_H1_branch01_recoarsened_to_H6.pdf
tree_H1_branch01_recoarsened_to_H6.png
tree_H1_branch01_relative_error_to_H6.pdf
tree_H1_branch01_relative_error_to_H6.png
tree_H1_branch02_generated.pdf
tree_H1_branch0